In [87]:
%pip install rawpy

Note: you may need to restart the kernel to use updated packages.


In [88]:
import rawpy
import imageio
import numpy as np
from matplotlib import pyplot as plt
import skimage
from scipy.ndimage import convolve
import scipy

In [89]:
# set input path
path_in = 'L1004432.DNG'

In [90]:
# filename for JPEG output
path_out  = path_in.split('.')[0] + '.jpg'
path_out

'L1004432.jpg'

In [91]:
# read 16-bit integer data from RAW file into raw_image array
with rawpy.imread(path_in) as raw:
    raw_image = raw.raw_image.copy()

### 1. Convert the raw image data to 32-bit floating point in [0 1] range.

In [92]:
raw_image_32 = raw_image.astype(np.float32) / np.iinfo(np.uint32).max

### 2. Bayer Filters

In [93]:
green = np.array([[0, 1/4, 0],
                  [1/4, 0, 1/4],
                  [0, 1/4, 0]])
diag = np.array([[1/4, 0, 1/4],
                 [0, 0, 0],
                 [1/4, 0, 1/4]])

horiz = np.array([[0, 0, 0],
                  [1/2, 0, 1/2],
                  [0, 0, 0]])

vert = np.array([[0, 1/2, 0],
                 [0, 0, 0],
                 [0, 1/2, 0]])

### 3. Apply Filter Kernels

In [94]:
bi_green = scipy.ndimage.correlate(raw_image_32, green, mode='mirror')
bi_diag  = scipy.ndimage.correlate(raw_image_32, diag, mode='mirror')
bi_horiz = scipy.ndimage.correlate(raw_image_32, horiz, mode='mirror')
bi_vert  = scipy.ndimage.correlate(raw_image_32, vert, mode='mirror')

### 4. Create a red , green , and blue array, each of the same shape as the RAW image.

In [95]:
green_channel = np.zeros(np.shape(raw_image_32))
blue_channel = np.zeros(np.shape(raw_image_32))
red_channel = np.zeros(np.shape(raw_image_32))

### 5. Copy values from RAW image or filter outputs

In [96]:

red_channel[0::2, 0::2] = raw_image_32[0::2, 0::2]
red_channel[0::2, 1::2] = bi_horiz[0::2, 1::2]
red_channel[1::2, 0::2] = bi_vert[1::2, 0::2]
red_channel[1::2, 1::2] = bi_diag[1::2, 1::2]


green_channel[0::2, 1::2] = raw_image_32[0::2, 1::2]
green_channel[1::2, 0::2] = raw_image_32[1::2, 0::2]
green_channel[0::2, 0::2] = bi_green[0::2, 0::2]
green_channel[1::2, 1::2] = bi_green[1::2, 1::2]


blue_channel[1::2, 1::2] = raw_image_32[1::2, 1::2]
blue_channel[0::2, 0::2] = bi_diag[0::2, 0::2]
blue_channel[0::2, 1::2] = bi_vert[0::2, 1::2]
blue_channel[1::2, 0::2] = bi_horiz[1::2, 0::2]

### 6. Stack the red, green, and blue channels

In [97]:
interpolated_image = np.stack([red_channel, green_channel, blue_channel], axis=-1)

### 7. Scale so mean is 0.25

In [98]:
interpolated_image[:,:,0] *= 0.25 / interpolated_image[:,:,0].mean()
interpolated_image[:,:,1] *= 0.25 / interpolated_image[:,:,1].mean()
interpolated_image[:,:,2] *= 0.25 / interpolated_image[:,:,2].mean()

### 8. Apply inverse gamma curve

In [99]:
interpolated_image = np.power(interpolated_image, 0.55)

### 9. Clip and quantize

In [100]:
interpolated_image = (np.clip(interpolated_image, 0, 1) * 255).astype(np.uint8)

### 10. Save image as output

In [101]:
imageio.imwrite(path_out,interpolated_image)